# 02 · Error analysis

Concrete examples behind the numbers in section 5 of `00_main.ipynb`: the question, the top 5 chunks from the chosen retriever with the judge's label, and what each model chose. The examples are drawn at random with a fixed seed, so they are not hand-picked and the notebook shows the same ones every time; change `n` or `seed` below to see others.

**Reads:** `results/raw/qa/`, `results/raw/runs/`, `results/raw/judge/`, `data/processed/` and `results/tables/retriever_choice.csv` (written by `00_main.ipynb`)

Groups:
1. Fixed: Gemma wrong without chunks, right with them, and a relevant chunk in the top 5
2. Broken: Gemma right without chunks, wrong with them, and no relevant chunk in the top 5
3. Wrong although a relevant chunk was in the top 5
4. Low letter mass: Gemma with chunks put less than half its probability on the option letters

This is material for the Discussion section; nothing here changes the results in `00_main.ipynb`.

In [1]:
# Picks up changes saved in src/ without restarting the notebook
%load_ext autoreload
%autoreload 2

import textwrap

import numpy as np
import pandas as pd

from norrag.config import get_path, load_config
from norrag.metrics import letter_mass, retrieval_scores
from norrag.records import read_jsonl_df

config = load_config()
k = config["qa"]["k"]
n = 2  # examples per group
seed = config["seed"]
raw = get_path(config, "results_raw")

# The retriever chosen in section 2 of 00_main.ipynb
chosen = pd.read_csv(get_path(config, "tables") / "retriever_choice.csv", index_col=0).loc["chosen", "value"]
retriever_names = {"bm25": "BM25", "dense": "sentence-BERT"}  # the same names as in 00_main.ipynb
gemma_rag = f"Gemma + {retriever_names[chosen]}"

questions = pd.read_parquet(get_path(config, "data_processed") / "questions.parquet").set_index("qid")
run = read_jsonl_df(raw / "runs" / f"{chosen}.jsonl")
judge = read_jsonl_df(raw / "judge" / f"{config['judge']['model']}.jsonl").rename(columns={"label": "relevant"})
qa = {
    "Borealis": read_jsonl_df(raw / "qa" / "borealis-12b__no_rag.jsonl").set_index("qid"),
    "Gemma": read_jsonl_df(raw / "qa" / "gemma-12b__no_rag.jsonl").set_index("qid"),
    gemma_rag: read_jsonl_df(raw / "qa" / f"gemma-12b__rag_{chosen}.jsonl").set_index("qid"),
}

# The texts and judge labels of every chunk in a top k
top_k = run.set_index("qid").chunk_ids.str[:k]
needed = sorted({chunk_id for ids in top_k for chunk_id in ids})
chunk_texts = pd.read_parquet(get_path(config, "data_processed") / "chunks.parquet", columns=["chunk_id", "text"],
                              filters=[("chunk_id", "in", needed)]).set_index("chunk_id")["text"]
judge_label = judge.set_index(["qid", "chunk_id"]).relevant

In [2]:
# One row per question: right or wrong for each condition, retrieval hit, and Gemma's letter mass with chunks
by_question = pd.DataFrame({name: results.correct for name, results in qa.items()})
by_question["hit"] = retrieval_scores(run, judge, k).set_index("qid").hit == 1
by_question["letter_mass"] = qa[gemma_rag].logprobs.apply(letter_mass)

groups = {
    "fixed": by_question[~by_question["Gemma"] & by_question[gemma_rag] & by_question.hit],
    "broken": by_question[by_question["Gemma"] & ~by_question[gemma_rag] & ~by_question.hit],
    "wrong despite a relevant chunk": by_question[~by_question[gemma_rag] & by_question.hit],
    "low letter mass": by_question[by_question.letter_mass < 0.5],
}
pd.Series({name: len(group) for name, group in groups.items()}, name="questions")

fixed                             434
broken                            335
wrong despite a relevant chunk     34
low letter mass                    98
Name: questions, dtype: int64

In [3]:
def show(qid):
    # Print one question: the options (✓ = correct), what each model chose, and the top k chunks with the judge's label
    question = questions.loc[qid]
    print("=" * 100)
    print(qid)
    print(textwrap.fill(question.question, 100))
    for label, option in zip(question.labels, question.options):
        print(f"  {'✓' if label == question.answer else ' '} {label}: {option}")
    print()
    for name, results in qa.items():
        result = results.loc[qid]
        probabilities = ", ".join(f"{label} {p:.0%}" for label, p in zip(question.labels, np.exp(result.logprobs)))
        print(f"  {name:13} chose {result.predicted} ({'right' if result.correct else 'wrong'})   "
              f"letters: {probabilities}   letter mass {letter_mass(result.logprobs):.0%}")
    print(f"\n  Top {k} chunks from {retriever_names[chosen]} (judge: Y = relevant, N = not):")
    for rank, chunk_id in enumerate(top_k[qid], start=1):
        mark = "Y" if judge_label[(qid, chunk_id)] == 1 else "N"
        print(textwrap.fill(chunk_texts[chunk_id], 100, initial_indent=f"  [{rank}] {mark}  ", subsequent_indent=" " * 9))
    print()

## 1 · Fixed by retrieval

Gemma got these wrong without chunks and right with them, and a relevant chunk was in the top 5.

In [4]:
for qid in groups["fixed"].sample(n, random_state=seed).index:
    show(qid)

1.15241360-4
Kunstverket «Mother and Child (Divided)» av Damien Hirst består av to dyr som er delt i to på langs.
Vet du hva slags dyr kunstneren har brukt i verket?
    A: Gris
    B: Hund
    C: Hval
  ✓ D: Ku

  Borealis      chose A (wrong)   letters: A 31%, B 9%, C 31%, D 24%   letter mass 94%
  Gemma         chose A (wrong)   letters: A 100%, B 0%, C 0%, D 0%   letter mass 100%
  Gemma + BM25  chose D (right)   letters: A 0%, B 0%, C 0%, D 100%   letter mass 100%

  Top 5 chunks from BM25 (judge: Y = relevant, N = not):
  [1] N  Damien Hirst en stigende stjerne etter at den berømte kunstsamleren Charles Saatchi
         ervervet en mengde av Hirst sine arbeider. Utstillinger i Norge I Norge har kunstverk av
         Damien Hirst vært utstilt i Stavanger, Oslo og i bygda Flø utenfor Ulsteinvik. Hans seneste
         salgsutstilling i Norge var høsten 2010 på Galleri Bi-Z i Kristiansand. I 2005 ble Hirst
         tildelt Turner Prize for verket «Mother and Child Divided», som tidli

## 2 · Broken by retrieval

Gemma got these right without chunks and wrong with them, and no chunk in the top 5 was relevant.

In [5]:
for qid in groups["broken"].sample(n, random_state=seed).index:
    show(qid)

1.14082565-12
Hvor i Norge finner du broen som skal ha gitt Peder Christen Asbjørnsen ideen til fortellingen om
"De Tre Bukkene Bruse"?
  ✓ A: Snertingdalen
    B: Gudbrandsdalen
    C: Lauvdal
    D: Groruddalen

  Borealis      chose B (wrong)   letters: A 13%, B 56%, C 21%, D 8%   letter mass 97%
  Gemma         chose A (right)   letters: A 94%, B 6%, C 0%, D 0%   letter mass 100%
  Gemma + BM25  chose B (wrong)   letters: A 0%, B 100%, C 0%, D 0%   letter mass 100%

  Top 5 chunks from BM25 (judge: Y = relevant, N = not):
  [1] N  De tre bukkene Bruse De tre bukkene Bruse eller De tre Bukkene Bruse, som skulde gaae til
         Sæters og gjøre sig fede er et kort, norsk folkeeventyr som Peter Christen Asbjørnsen fikk
         høre fortalt av Anders Lysgaard fra Svennes Gård i Biri i 1842. Eventyret ble første gang
         gjengitt i tredje hefte av Norske Folkeeventyr (1843). Eventyret handler om tre geitebukker
         som på vei til sommerbeite på en seter blir truet av et trol

## 3 · Wrong although a relevant chunk was found

A chunk in the top 5 gave the answer, but Gemma still chose a wrong option.

In [6]:
for qid in groups["wrong despite a relevant chunk"].sample(n, random_state=seed).index:
    show(qid)

1.14977709-1
Hvor mange appelsiner spiser nordmenn i påska?
    A: 15 millioner
  ✓ B: 16 millioner
    C: 17 millioner
    D: 18 millioner

  Borealis      chose A (wrong)   letters: A 25%, B 25%, C 25%, D 25%   letter mass 98%
  Gemma         chose B (right)   letters: A 3%, B 77%, C 17%, D 2%   letter mass 99%
  Gemma + BM25  chose C (wrong)   letters: A 0%, B 0%, C 100%, D 0%   letter mass 100%

  Top 5 chunks from BM25 (judge: Y = relevant, N = not):
  [1] Y  Appelsin kroner, 75 % av disse fra Spania. Dette var en tilbakegang på ca. 5 % i forhold
         til året før. Bruk Konsum i Norge I Norge spises stort sett appelsiner ferske i båter, men
         de kan også brukes i salater, sauser, marinader og dressinger. Også skallet kan brukes, for
         eksempel er apsikat laget av appelsinskall. Hver påske spises det mer enn 17 millioner
         appelsiner i Norge. Oppbevaring Appelsinen har god holdbarhetstid og oppbevares best ved
         5–10 °C, ikke i kjøleskap.
  [2] N  Me

## 4 · Low letter mass

Gemma with chunks put less than half its probability on the option letters: it would rather have started its answer with something else, so the chosen letter is close to a guess.

In [7]:
for qid in groups["low letter mass"].sample(n, random_state=seed).index:
    show(qid)

1.15392659-3b
I februar 2021 var det smitteutbrudd ved byggeplassen til Petter Stordalens nye hotell i Oslo. Hvor
mange var bekreftet smittet?
    A: 12
    B: 22
  ✓ C: 32

  Borealis      chose C (right)   letters: A 11%, B 14%, C 72%   letter mass 97%
  Gemma         chose C (right)   letters: A 0%, B 0%, C 100%   letter mass 100%
  Gemma + BM25  chose C (right)   letters: A 2%, B 2%, C 19%   letter mass 23%

  Top 5 chunks from BM25 (judge: Y = relevant, N = not):
  [1] N  Petter Stordalen innen eiendom, finans, hotelldrift og kunst og kapitalforvaltning.
         Strawberry består i 2021 av Nordic Choice Hotels og øvrige datterselskaper. Karriere Petter
         Stordalen er sønn av tidligere norgesmester i bryting Knut Anker Stordalen, og vokste opp i
         Porsgrunn. Faren var fra gården Stordalen på Bjørnholt i Jondalen i nåværende Kongsberg
         kommune, der familienavnet skriver seg fra, og mange av forfedrene jobbet i generasjoner
         ved Kongsberg Sølvverk. Pett